# YOLO11-seg 直肠肿瘤分割训练

**环境**: Kaggle T4 GPU

**数据集**: 上传 `rectal_tumor_seg_dataset.zip` 作为 Kaggle Dataset

## 操作步骤
1. 在 Kaggle 新建 Dataset，上传 `rectal_tumor_seg_dataset.zip`
2. 新建 Notebook，添加该 Dataset，打开 GPU (T4 x2)
3. 运行本 notebook 所有 cell

In [ ]:
# 1. 安装/升级 ultralytics
!pip install -q ultralytics>=8.3.0

In [ ]:
import os, shutil, torch
from ultralytics import YOLO

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# 2. 解压数据集
# Kaggle dataset 路径: /kaggle/input/<dataset-name>/
# 请根据你上传的 dataset 名称修改下面的路径

import glob

# 自动查找 zip 文件
zip_candidates = glob.glob('/kaggle/input/**/rectal_tumor_seg_dataset.zip', recursive=True)
if not zip_candidates:
    zip_candidates = glob.glob('/kaggle/input/**/*.zip', recursive=True)

if zip_candidates:
    zip_path = zip_candidates[0]
    print(f'Found zip: {zip_path}')
else:
    # 如果是直接解压上传的，检查目录
    print('No zip found, checking for extracted dataset...')
    zip_path = None

WORK_DIR = '/kaggle/working'
DATA_DIR = os.path.join(WORK_DIR, 'datasets', 'rectal_tumor_seg')

if zip_path and not os.path.exists(DATA_DIR):
    print('Extracting...')
    import zipfile
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(WORK_DIR)
    print('Done')
elif not os.path.exists(DATA_DIR):
    # Kaggle 可能自动解压到 input 目录
    input_data = glob.glob('/kaggle/input/**/images/train', recursive=True)
    if input_data:
        src_root = os.path.dirname(os.path.dirname(input_data[0]))
        print(f'Copying from {src_root} ...')
        shutil.copytree(src_root, DATA_DIR)
        print('Done')

# 验证
train_imgs = len(os.listdir(os.path.join(DATA_DIR, 'images', 'train')))
val_imgs = len(os.listdir(os.path.join(DATA_DIR, 'images', 'val')))
print(f'Train: {train_imgs}, Val: {val_imgs}')

In [ ]:
# 3. 修正 data.yaml 路径 (适配 Kaggle 环境)
yaml_path = os.path.join(DATA_DIR, 'data.yaml')

yaml_content = f"""# CTAI rectal tumor segmentation dataset
path: {DATA_DIR}
train: images/train
val: images/val

names:
  0: tumor

nc: 1
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f'data.yaml updated: {yaml_path}')
print(yaml_content)

In [ ]:
# 4. 开始训练 YOLO11-seg (双 T4 GPU)
model = YOLO('yolo11n-seg.pt')  # 自动下载预训练权重

# 检测 GPU 数量，自动适配
gpu_count = torch.cuda.device_count()
if gpu_count >= 2:
    device = [0, 1]
    batch = 32        # 双 T4 可以跑 batch=32
    print(f'[INFO] 双 GPU 模式, batch={batch}')
else:
    device = 0
    batch = 16
    print(f'[INFO] 单 GPU 模式, batch={batch}')

results = model.train(
    data=yaml_path,
    epochs=200,
    imgsz=512,
    batch=batch,
    patience=30,          # 早停
    save=True,
    save_period=10,
    device=device,        # 双 GPU 并行
    workers=4,
    project='runs/segment',
    name='rectal_tumor',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    close_mosaic=10,
    # 医学图像增强策略
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    degrees=15.0,
    translate=0.1,
    scale=0.3,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.0,
    overlap_mask=True,
    mask_ratio=4,
    single_cls=True,
)

print(f'\nBest model: {results.save_dir}/weights/best.pt')

In [ ]:
# 5. 查看训练结果
from IPython.display import Image, display

save_dir = 'runs/segment/rectal_tumor'

# 训练曲线
if os.path.exists(f'{save_dir}/results.png'):
    display(Image(filename=f'{save_dir}/results.png', width=900))

# 混淆矩阵
if os.path.exists(f'{save_dir}/confusion_matrix.png'):
    display(Image(filename=f'{save_dir}/confusion_matrix.png', width=600))

In [ ]:
# 6. 在验证集上评估，获取详细指标
best_model = YOLO(f'{save_dir}/weights/best.pt')
metrics = best_model.val(
    data=yaml_path,
    imgsz=512,
    batch=16,
    device=0,
    split='val',
)

print('\n========== 验证集指标 ==========')
print(f'mAP@50 (mask):    {metrics.seg.map50:.4f}')
print(f'mAP@50-95 (mask): {metrics.seg.map:.4f}')
print(f'Precision:        {metrics.seg.mp:.4f}')
print(f'Recall:           {metrics.seg.mr:.4f}')
print(f'\nmAP@50 (box):     {metrics.box.map50:.4f}')
print(f'mAP@50-95 (box):  {metrics.box.map:.4f}')

In [ ]:
# 7. 计算 Dice 系数 (从验证集推理结果)
import numpy as np
import cv2

val_img_dir = os.path.join(DATA_DIR, 'images', 'val')
val_lbl_dir = os.path.join(DATA_DIR, 'labels', 'val')

dice_scores = []
inference_times = []

val_images = sorted([f for f in os.listdir(val_img_dir) if f.endswith('.png')])

for img_name in val_images:
    img_path = os.path.join(val_img_dir, img_name)
    lbl_name = img_name.replace('.png', '.txt')
    lbl_path = os.path.join(val_lbl_dir, lbl_name)
    
    # 读取图像
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    
    # 构建 GT mask
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 7:  # class_id + at least 3 points
                    continue
                coords = list(map(float, parts[1:]))
                pts = []
                for i in range(0, len(coords), 2):
                    px = int(coords[i] * w)
                    py = int(coords[i+1] * h)
                    pts.append([px, py])
                pts = np.array(pts, dtype=np.int32)
                cv2.fillPoly(gt_mask, [pts], 255)
    
    # YOLO 推理
    import time
    t0 = time.time()
    results = best_model.predict(source=img, conf=0.25, iou=0.45, imgsz=512,
                                  retina_masks=True, verbose=False)
    t1 = time.time()
    inference_times.append((t1 - t0) * 1000)  # ms
    
    # 构建预测 mask
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if results[0].masks is not None:
        for mask in results[0].masks.data.cpu().numpy():
            if mask.shape[:2] != (h, w):
                mask = cv2.resize(mask, (w, h))
            pred_mask = np.maximum(pred_mask, (mask > 0.5).astype(np.uint8) * 255)
    
    # 计算 Dice
    gt_bin = (gt_mask > 0).astype(np.float32)
    pred_bin = (pred_mask > 0).astype(np.float32)
    intersection = (gt_bin * pred_bin).sum()
    union_sum = gt_bin.sum() + pred_bin.sum()
    
    if union_sum == 0:
        dice = 1.0  # both empty
    else:
        dice = 2 * intersection / union_sum
    
    dice_scores.append(dice)

# 只统计有肿瘤的切片的 Dice
tumor_dices = [d for d in dice_scores if d < 1.0]  # exclude both-empty cases
all_dices = dice_scores

print(f'\n========== Dice 系数 ==========')
print(f'全部验证集 Dice (含空切片): {np.mean(all_dices):.4f} +/- {np.std(all_dices):.4f}')
if tumor_dices:
    print(f'含肿瘤切片 Dice:           {np.mean(tumor_dices):.4f} +/- {np.std(tumor_dices):.4f}')
    print(f'含肿瘤切片数:              {len(tumor_dices)}')
print(f'\n========== 推理速度 ==========')
print(f'平均推理时间: {np.mean(inference_times):.1f} ms')
print(f'FPS: {1000 / np.mean(inference_times):.1f}')

In [ ]:
# 8. 可视化分割结果 (选几个典型 case)
import matplotlib.pyplot as plt

# 找含肿瘤的验证图片
tumor_imgs = []
for img_name in val_images:
    lbl_path = os.path.join(val_lbl_dir, img_name.replace('.png', '.txt'))
    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        tumor_imgs.append(img_name)

# 选 4 个展示
show_imgs = tumor_imgs[:4] if len(tumor_imgs) >= 4 else tumor_imgs

fig, axes = plt.subplots(len(show_imgs), 4, figsize=(16, 4 * len(show_imgs)))
if len(show_imgs) == 1:
    axes = [axes]

for idx, img_name in enumerate(show_imgs):
    img_path = os.path.join(val_img_dir, img_name)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # GT mask
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    lbl_path = os.path.join(val_lbl_dir, img_name.replace('.png', '.txt'))
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 7:
                    continue
                coords = list(map(float, parts[1:]))
                pts = [[int(coords[i]*w), int(coords[i+1]*h)] for i in range(0, len(coords), 2)]
                cv2.fillPoly(gt_mask, [np.array(pts, np.int32)], 255)
    
    # Pred mask
    results = best_model.predict(source=img, conf=0.25, iou=0.45, imgsz=512,
                                  retina_masks=True, verbose=False)
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    if results[0].masks is not None:
        for mask in results[0].masks.data.cpu().numpy():
            if mask.shape[:2] != (h, w):
                mask = cv2.resize(mask, (w, h))
            pred_mask = np.maximum(pred_mask, (mask > 0.5).astype(np.uint8) * 255)
    
    # Overlay
    overlay = img_rgb.copy()
    overlay[pred_mask > 0] = overlay[pred_mask > 0] * 0.6 + np.array([0, 255, 0]) * 0.4
    overlay = overlay.astype(np.uint8)
    
    axes[idx][0].imshow(img_rgb, cmap='gray')
    axes[idx][0].set_title('CT Original')
    axes[idx][0].axis('off')
    
    axes[idx][1].imshow(gt_mask, cmap='gray')
    axes[idx][1].set_title('Ground Truth')
    axes[idx][1].axis('off')
    
    axes[idx][2].imshow(pred_mask, cmap='gray')
    axes[idx][2].set_title('YOLO11 Prediction')
    axes[idx][2].axis('off')
    
    axes[idx][3].imshow(overlay)
    axes[idx][3].set_title('Overlay')
    axes[idx][3].axis('off')

plt.tight_layout()
plt.savefig('val_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: val_visualization.png')

In [ ]:
# 9. 模型信息 (论文表格用)
import yaml

# 参数量
model_info = best_model.info(verbose=False)

best_pt = f'{save_dir}/weights/best.pt'
pt_size = os.path.getsize(best_pt) / 1024 / 1024

print('\n========== 论文数据汇总 ==========')
print(f'模型文件大小: {pt_size:.1f} MB')
print(f'\n--- 表5-3: YOLO11-seg 验证集性能 ---')
print(f'mAP@50 (mask):    {metrics.seg.map50:.4f}')
print(f'mAP@50-95 (mask): {metrics.seg.map:.4f}')
print(f'Precision:        {metrics.seg.mp:.4f}')
print(f'Recall:           {metrics.seg.mr:.4f}')
print(f'Dice (肿瘤切片):  {np.mean(tumor_dices):.4f}' if tumor_dices else 'N/A')
print(f'推理时间:         {np.mean(inference_times):.1f} ms')
print(f'FPS:              {1000/np.mean(inference_times):.1f}')
print(f'\n--- 表5-2: 数据集统计 ---')
print(f'训练集图像数: {train_imgs}')
print(f'验证集图像数: {val_imgs}')
print(f'总图像数: {train_imgs + val_imgs}')

In [ ]:
# 10. 下载最佳权重和训练曲线
# Kaggle 训练完成后，从 Output 下载以下文件:

# 复制关键文件到 output 根目录方便下载
for src_file in [
    f'{save_dir}/weights/best.pt',
    f'{save_dir}/weights/last.pt',
    f'{save_dir}/results.png',
    f'{save_dir}/results.csv',
    f'{save_dir}/confusion_matrix.png',
    'val_visualization.png',
]:
    if os.path.exists(src_file):
        dst = os.path.join('/kaggle/working', os.path.basename(src_file))
        shutil.copy2(src_file, dst)
        print(f'Copied: {dst}')

print('\n训练完成！请下载以下文件:')
print('  best.pt        -> 复制到 CTAI_flask/core/net/best.pt')
print('  results.png    -> 论文图5-1, 5-2')
print('  results.csv    -> 备用数据')
print('  val_visualization.png -> 论文图5-3')